## Score-based Generative Modeling through Stochastic Differential Equation

This notebook provides an example of a Score-based Generative model that uses an SDE for generation. The model is based on [Song et al. (2021)](https://github.com/yang-song/score_sde_pytorch) and [Song (2021)](https://colab.research.google.com/drive/120kYYBOVa1i0TD85RjlEkFjaWDxSFUx3?usp=sharing#scrollTo=XCR6m0HjWGVV).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.optim import Adam
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torchvision.utils import make_grid, save_image
#Functools
import functools
# tqdm
from tqdm.notebook import tqdm, trange

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Load and Prepare CIFAR-10 Dataset

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
cifar10_trainset = CIFAR10(root='./data', train=True, download=True, transform=transform)

In [ ]:
train_batch_size = 128
train_loader = data.DataLoader(cifar10_trainset, batch_size=train_batch_size, 
                               shuffle=True,  drop_last=True, pin_memory=True)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Define CNN model for the Score
The Score is a function that takes an image as input and outputs a structure of the same size containing the score. Hence it requires an autocoder like structure for the model. Here a U-Net based on ResNet blocks is used and the architecture is based on [Yang and Ermon (2019)](https://arxiv.org/pdf/1907.05600.pdf)

In [ ]:
class GaussianFourierProjection(nn.Module):
    def __init__(self, embed_dim, scale=30.):
        super().__init__()
        self.W = nn.Parameter(torch.randn(embed_dim // 2) * scale, requires_grad=False)
  
    def forward(self, x):
        x_proj = x[:, None] * self.W[None, :] * 2 * np.pi
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)

class Dense(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, output_dim)
    def forward(self, x):
        return self.dense(x)[..., None, None]

In [ ]:
def swish(x):
    return x * torch.sigmoid(x)

In [ ]:
class ScoreNet(nn.Module):
    def __init__(self, marginal_prob_std, embed_dim=1024):
        super().__init__()
    
        # Gaussian random feature embedding layer for time
        self.embed = nn.Sequential(GaussianFourierProjection(embed_dim=embed_dim),
                                   nn.Linear(embed_dim, embed_dim))
        # Encoding layers where the resolution decreases
        self.conv1 = nn.Conv2d(3, 64, 3, stride=2, padding=1, bias=False)
        self.dense1 = Dense(embed_dim, 64)
        self.gnorm1 = nn.GroupNorm(4, num_channels=64)
        self.conv2 = nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)
        self.dense2 = Dense(embed_dim, 128)
        self.gnorm2 = nn.GroupNorm(32, num_channels=128)
        self.conv3 = nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False)
        self.dense3 = Dense(embed_dim, 256)
        self.gnorm3 = nn.GroupNorm(32, num_channels=256)
        self.conv4 = nn.Conv2d(256, 512, 3, stride=2, padding=1, bias=False)
        self.dense4 = Dense(embed_dim, 512)
        self.gnorm4 = nn.GroupNorm(32, num_channels=512)    

        # Decoding layers where the resolution increases
        self.tconv4 = nn.ConvTranspose2d(512, 256, 3, stride=2, padding=1, output_padding = 1, bias=False)
        self.dense5 = Dense(embed_dim, 256)
        self.tgnorm4 = nn.GroupNorm(32, num_channels=256)
        self.tconv3 = nn.ConvTranspose2d(512, 128, 3, stride=2, padding=1, output_padding = 1, bias=False)    
        self.dense6 = Dense(embed_dim, 128)
        self.tgnorm3 = nn.GroupNorm(32, num_channels=128)
        self.tconv2 = nn.ConvTranspose2d(256, 64, 3, stride=2, padding=1, output_padding = 1, bias=False)    
        self.dense7 = Dense(embed_dim, 64)
        self.tgnorm2 = nn.GroupNorm(32, num_channels=64)
        self.tconv1 = nn.ConvTranspose2d(128, 3, 3, stride=2, padding=1, output_padding = 1, bias=False)
        self.marginal_prob_std = marginal_prob_std

    def forward(self, x, t): 
        # Obtain the Gaussian random feature embedding for t   
        embed = swish(self.embed(t))    
        # Encoding path
        h1 = self.conv1(x)    
        ## Incorporate information from t
        h1 += self.dense1(embed)
        ## Group normalization
        h1 = swish(self.gnorm1(h1))
        h2 = self.conv2(h1)
        h2 += self.dense2(embed)
        h2 = swish(self.gnorm2(h2))
        h3 = self.conv3(h2)
        h3 += self.dense3(embed)
        h3 = swish(self.gnorm3(h3))
        h4 = self.conv4(h3)
        h4 += self.dense4(embed)
        h4 = swish(self.gnorm4(h4))

        # Decoding path
        h = self.tconv4(h4)
        ## Skip connection from the encoding path
        h += self.dense5(embed)
        h = swish(self.tgnorm4(h))
        h = self.tconv3(torch.cat([h, h3], dim=1))
        h += self.dense6(embed)
        h = swish(self.tgnorm3(h))
        h = self.tconv2(torch.cat([h, h2], dim=1))
        h += self.dense7(embed)
        h = swish(self.tgnorm2(h))
        h = self.tconv1(torch.cat([h, h1], dim=1))
        # Normalize output
        h = h / self.marginal_prob_std(t)[:, None, None, None]
        return h

### Set up the SDE

Define a function that will return the standard deviation of $p_{0t}(x(t) | x(0))$.

In [ ]:
def marginal_prob_std(t, sigma):
    t = torch.tensor(t, device=device)
    return torch.sqrt((sigma**(2 * t) - 1.) / 2. / np.log(sigma))

Define a function that calculates the SDE diffusion coefficient

In [ ]:
def diffusion_coeff(t, sigma):
    return torch.tensor(sigma**t, device=device)

In [ ]:
sigma =  25.0
marginal_prob_std_fn = functools.partial(marginal_prob_std, sigma=sigma)
diffusion_coeff_fn = functools.partial(diffusion_coeff, sigma=sigma)

### Define Sampler
The sampler functions manage the sampling process by solving the reverse Stochastic Differential Equation. This is done here using an Euler-Maruyama discrete SDE approximation

In [ ]:
def Euler_Maruyama_sampler(score_model, 
                           marginal_prob_std,
                           diffusion_coeff,
                           batch_size=64, 
                           num_steps=512,
                           return_img_per_step=False,
                           eps=1e-3):
    t = torch.ones(batch_size, device=device)
    init_x = torch.randn(batch_size, 3, 32, 32, device=device) \
            * marginal_prob_std(t)[:, None, None, None]
    time_steps = torch.linspace(1., eps, num_steps, device=device)
    step_size = time_steps[0] - time_steps[1]
    x = init_x
    
    # List for storing generations at each step
    imgs_per_step = []
    
    with torch.no_grad():
        for time_step in tqdm(time_steps):
            batch_time_step = torch.ones(batch_size, device=device) * time_step
            g = diffusion_coeff(batch_time_step)
            mean_x = x + (g**2)[:, None, None, None] \
                     * score_model(x, batch_time_step) * step_size
            x = mean_x + torch.sqrt(step_size) * g[:, None, None, None] \
                     * torch.randn_like(x)
            if return_img_per_step:
                imgs_per_step.append(mean_x.clone().detach())
    
    if return_img_per_step:
        return torch.stack(imgs_per_step, dim=0).cpu()
    
    return mean_x

### Train the Model

In [ ]:
def loss_fn(model, x, marginal_prob_std, eps=1e-5):
    random_t = torch.rand(x.shape[0], device=device) * (1. - eps) + eps  
    z = torch.randn_like(x)
    std = marginal_prob_std(random_t)
    perturbed_x = x + z * std[:, None, None, None]
    score = model(perturbed_x, random_t)
    loss = torch.mean(torch.sum((score * std[:, None, None, None] + z)**2, 
                                dim=(1,2,3)))
    return loss

In [ ]:
def sample_save(score_model, batch_size, filepath, epoch, blocks = 16):
    epoch_path = os.path.join(filepath, f"epoch_{epoch}")
    if not os.path.exists(epoch_path):
        os.makedirs(epoch_path)
        
    block_size = batch_size // blocks
    for iblk in range(blocks):
        idx_start = iblk * block_size
        idx_end = (iblk + 1) * block_size
    
        imgs = Euler_Maruyama_sampler(score_model, 
                                      marginal_prob_std_fn,
                                      diffusion_coeff_fn,
                                      batch_size=block_size)
          
        # Generate images
        for idx in range(idx_start, idx_end):
            filename = os.path.join(epoch_path, f"generation_{idx}.png")
            torchvision.utils.save_image(imgs[idx - idx_start], filename)

In [ ]:
score_model = ScoreNet(marginal_prob_std=marginal_prob_std_fn)
score_model = score_model.to(device)

n_epochs = 3000
lr=2e-4
sample_freq = 300
sample_size = 32768
checkpoint_path_cifar10 = './Score/check_point/CIFAR10'
os.makedirs(checkpoint_path_cifar10, exist_ok=True)

optimizer = Adam(score_model.parameters(), lr=lr)
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_loss = 0.
    num_items = 0
    for x, y in train_loader:
        x = x.to(device)    
        loss = loss_fn(score_model, x, marginal_prob_std_fn)
        optimizer.zero_grad()
        loss.backward()    
        optimizer.step()
        avg_loss += loss.item() * x.shape[0]
        num_items += x.shape[0]
    
    chkpoint_file = os.path.join(checkpoint_path_cifar10, f'score_cifar10_ckpt_{epoch + 1}.pth')
    tqdm_epoch.set_description('Average Loss: {:5f}'.format(avg_loss / num_items))
    torch.save(score_model.state_dict(), chkpoint_file)
    if (epoch + 1) % sample_freq == 0:
        sample_save(score_model, sample_size, checkpoint_path_cifar10, epoch + 1)

### Sample

In [ ]:
imgs_per_step = Euler_Maruyama_sampler(score_model, marginal_prob_std_fn,
                                       diffusion_coeff_fn,
                                       batch_size=4, num_steps=512, 
                                       return_img_per_step=True)

In [ ]:
vis_steps=8
num_steps=512
for i in range(imgs_per_step.shape[1]):
    step_size = num_steps // vis_steps
    imgs_to_plot = imgs_per_step[step_size-1::step_size,i]
    imgs_to_plot = torch.cat([imgs_per_step[0:1,i],imgs_to_plot], dim=0)
    grid = torchvision.utils.make_grid(imgs_to_plot, 
                                       nrow=imgs_to_plot.shape[0], 
                                       normalize=True, value_range=(-1,1), 
                                       pad_value=0.5, padding=2)
    grid = grid.permute(1, 2, 0)
    plt.figure(figsize=(8,8))
    plt.imshow(grid)
    plt.xlabel("Generation iteration")
    plt.xticks([(imgs_per_step.shape[-1]+2)*(0.5+j) for j in range(vis_steps+1)],
               labels=[1] + list(range(step_size,imgs_per_step.shape[0]+1,step_size)))
    plt.yticks([])
    plt.savefig('ScoreGenerationCIFAR10It_' + str(i) + '.png', dpi=300, bbox_inches='tight')

In [ ]:
sample_batch_size = 64

samples = Euler_Maruyama_sampler(score_model, 
                          marginal_prob_std_fn,
                          diffusion_coeff_fn, 
                          sample_batch_size)

## Sample visualization.
samples = samples.clamp(0.0, 1.0)

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(samples,
                   num_images=64, 
                   size=(3, 32, 32), 
                   nrow=8,  
                   filename='ScoreGenerationCIFAR10Grid.png')